## PART A — Probability Foundations
## Task 1: Conditional Probability

In [1]:
# Given probabilities
P_A = 0.01        # Probability of having the disease
P_B_given_A = 0.95  # Probability test is positive given disease
P_B = 0.10        # Probability of testing positive

# Bayes' Theorem
P_A_given_B = (P_B_given_A * P_A) / P_B

print("Probability of having the disease given a positive test:", P_A_given_B)


Probability of having the disease given a positive test: 0.09499999999999999


## PART B — Naive Bayes from Scratch (Core Task)
## Task 2: Dataset Selection

Chosen Dataset: Text Data (Spam vs Ham)

Dataset type:
Text classification (binary)

Classes:

Spam (1) → Unwanted / promotional messages
Ham (0) → Normal / genuine messages

In [2]:
## Example dataset:

texts = [
    "Win a free iPhone now",
    "Limited offer claim prize",
    "Meeting at 10 am tomorrow",
    "Let's have lunch today",
    "Congratulations you won a lottery",
    "Are we still on for the project discussion"
]

labels = [
    1,  # spam
    1,  # spam
    0,  # ham
    0,  # ham
    1,  # spam
    0   # ham
]


 ## Task 3: Implement Naive Bayes (From Scratch)
 ## Gaussian Naive Bayes

In [3]:
## Example dataset
# One feature: marks

X = [30, 35, 40, 60, 65, 70]
y = [0, 0, 0, 1, 1, 1]   # 0 = Fail, 1 = Pass


In [ ]:
 ## Separate Data by Class

import math

X_0 = [x for x, label in zip(X, y) if label == 0]
X_1 = [x for x, label in zip(X, y) if label == 1]


In [5]:
 ## Mean

def mean(values):
    return sum(values) / len(values)

mean_0 = mean(X_0)
mean_1 = mean(X_1)


In [6]:
## Variance

def variance(values, mean):
    return sum((x - mean) ** 2 for x in values) / len(values)

var_0 = variance(X_0, mean_0)
var_1 = variance(X_1, mean_1)


In [7]:
## Likelihood

def gaussian_likelihood(x, mean, var):
    exponent = math.exp(-((x - mean) ** 2) / (2 * var))
    return (1 / math.sqrt(2 * math.pi * var)) * exponent


In [8]:
## Prior Probability

P_0 = len(X_0) / len(X)
P_1 = len(X_1) / len(X)

In [9]:
## Posterior Probability and Prediction

def predict(x):
    prob_0 = gaussian_likelihood(x, mean_0, var_0) * P_0
    prob_1 = gaussian_likelihood(x, mean_1, var_1) * P_1

    return 1 if prob_1 > prob_0 else 0


In [10]:
## Test the Model

test_value = 50
result = predict(test_value)

print("Pass" if result == 1 else "Fail")

Fail


## Multinomial Naive Bayes

In [11]:
## Example dataset

texts = [
    "free money win",
    "win a prize",
    "meeting tomorrow",
    "lets have lunch",
    "free lottery win",
    "project meeting"
]

labels = [1, 1, 0, 0, 1, 0]

In [14]:
## Tokenization

def tokenize(text):
    return text.lower().split()

## Example

"Free Money Win" == ["free", "money", "win"]


False

In [15]:
## Word counts

from collections import defaultdict

spam_counts = defaultdict(int)
ham_counts = defaultdict(int)

spam_docs = 0
ham_docs = 0

for text, label in zip(texts, labels):
    words = tokenize(text)
    
    if label == 1:
        spam_docs += 1
        for w in words:
            spam_counts[w] += 1
    else:
        ham_docs += 1
        for w in words:
            ham_counts[w] += 1


In [16]:
## Prior Probability

total_docs = len(texts)

P_spam = spam_docs / total_docs
P_ham = ham_docs / total_docs

In [17]:
## Likelihood

vocab = set(spam_counts.keys()) | set(ham_counts.keys())
V = len(vocab)

spam_total_words = sum(spam_counts.values())
ham_total_words = sum(ham_counts.values())

def likelihood(word, word_counts, total_words):
    return (word_counts[word] + 1) / (total_words + V)

In [18]:
## Prediction (Posterior Probability)

import math

def predict(text):
    words = tokenize(text)

    log_spam = math.log(P_spam)
    log_ham = math.log(P_ham)

    for w in words:
        log_spam += math.log(likelihood(w, spam_counts, spam_total_words))
        log_ham += math.log(likelihood(w, ham_counts, ham_total_words))

    return 1 if log_spam > log_ham else 0


In [19]:
## Test the Model

msg = "free win money"
print("Spam" if predict(msg) == 1 else "Ham")

Spam


## Task 4: Prediction Logic

In [20]:
## Example test message

test_message = "free win money"

## Compute Class Probabilities

import math

def predict_with_details(text):
    words = tokenize(text)

    log_spam = math.log(P_spam)
    log_ham = math.log(P_ham)

    print("Message:", text)
    print("Prior Spam:", P_spam)
    print("Prior Ham:", P_ham)
    print()

    for w in words:
        spam_likelihood = likelihood(w, spam_counts, spam_total_words)
        ham_likelihood = likelihood(w, ham_counts, ham_total_words)

        print(f"Word: '{w}'")
        print("  P(word | spam):", spam_likelihood)
        print("  P(word | ham) :", ham_likelihood)

        log_spam += math.log(spam_likelihood)
        log_ham += math.log(ham_likelihood)

    print("\nLog Posterior Spam:", log_spam)
    print("Log Posterior Ham :", log_ham)

    if log_spam > log_ham:
        print("Prediction: Spam")
        return 1
    else:
        print("Prediction: Ham")
        return 0


In [22]:
## Minimal Working Code (Intermediate Probabilities + Prediction)

import math

# Example test message
test_message = "free win money"

# Initialize log priors
log_spam = math.log(P_spam)
log_ham = math.log(P_ham)

print("Test message:", test_message)
print("Initial log prior (Spam):", log_spam)
print("Initial log prior (Ham) :", log_ham)
print()

# Go through each word and add log-likelihood
for word in tokenize(test_message):
    spam_l = likelihood(word, spam_counts, spam_total_words)
    ham_l = likelihood(word, ham_counts, ham_total_words)

    print(f"Word: '{word}'")
    print("  P(word | Spam):", spam_l)
    print("  P(word | Ham) :", ham_l)

    log_spam += math.log(spam_l)
    log_ham += math.log(ham_l)

# Now final posterior probabilities (in log)
print("\nFinal log posterior (Spam):", log_spam)
print("Final log posterior (Ham) :", log_ham)

# Select class with maximum posterior
prediction = "Spam" if log_spam > log_ham else "Ham"
print("\nPredicted class:", prediction)


Test message: free win money
Initial log prior (Spam): -0.6931471805599453
Initial log prior (Ham) : -0.6931471805599453

Word: 'free'
  P(word | Spam): 0.14285714285714285
  P(word | Ham) : 0.05263157894736842
Word: 'win'
  P(word | Spam): 0.19047619047619047
  P(word | Ham) : 0.05263157894736842
Word: 'money'
  P(word | Spam): 0.09523809523809523
  P(word | Ham) : 0.05263157894736842

Final log posterior (Spam): -6.64866066338227
Final log posterior (Ham) : -9.526464118059268

Predicted class: Spam


## PART C — Evaluation
## Task 5: Metrics

In [23]:
## Example predictions & True labels

# True labels
y_true = [1, 0, 1, 0, 1, 0]

# Predictions from our Naive Bayes
y_pred = [1, 0, 1, 0, 0, 0]  # example


In [24]:
## Accuracy

accuracy = sum(yt == yp for yt, yp in zip(y_true, y_pred)) / len(y_true)
print("Accuracy:", accuracy)

## Precision

true_positive = sum((yt == 1 and yp == 1) for yt, yp in zip(y_true, y_pred))
predicted_positive = sum(yp == 1 for yp in y_pred)

precision = true_positive / predicted_positive if predicted_positive != 0 else 0
print("Precision:", precision)

## Recall

actual_positive = sum(yt == 1 for yt in y_true)

recall = true_positive / actual_positive if actual_positive != 0 else 0
print("Recall:", recall)


Accuracy: 0.8333333333333334
Precision: 1.0
Recall: 0.6666666666666666


## PART D — Experiments & Insights
## Task 7: sklearn Comparison

In [25]:
## Prepare the Data

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

texts = [
    "free money win",
    "win a prize",
    "meeting tomorrow",
    "lets have lunch",
    "free lottery win",
    "project meeting"
]

labels = [1, 1, 0, 0, 1, 0]

# Convert text to word count vectors
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)


In [26]:
## Train sklearn MultinomialNB

model = MultinomialNB()
model.fit(X, labels)

# Predictions
y_pred_sklearn = model.predict(X)
print("Predictions (sklearn):", y_pred_sklearn)

# Accuracy
accuracy = accuracy_score(labels, y_pred_sklearn)
print("Accuracy (sklearn):", accuracy)


Predictions (sklearn): [1 1 0 0 1 0]
Accuracy (sklearn): 1.0


In [27]:
## Compare With Scratch Implementation

# Predictions from your scratch Naive Bayes
y_pred_scratch = [predict(text) for text in texts]

# Accuracy
accuracy_scratch = sum(yt == yp for yt, yp in zip(labels, y_pred_scratch)) / len(labels)
print("Accuracy (scratch):", accuracy_scratch)


Accuracy (scratch): 1.0
